In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH = Path("/kaggle/input/datasets/bapdesilva/betel-dataset-bp/Betel Leaf Image Dataset from Bangladesh/Augmented Images/Train")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")   # offline-augmented copy
FINAL_PATH     = Path("/kaggle/working/final_dataset")        # train/val split lives here

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2000   # augment up to this count before any training
VAL_SPLIT        = 0.20   # 80/20 train-val split
IMG_SIZE         = 224    # EfficientNetB0 native resolution

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE      = 32
NUM_EPOCHS      = 30
LR_HEAD         = 1e-3    # Phase-1: only classifier head
LR_FINE         = 1e-4    # Phase-2: fine-tune unlocked layers
WEIGHT_DECAY    = 1e-4
PATIENCE        = 7       # early-stopping patience (per phase)
UNFREEZE_PCT    = 0.30    # fraction of backbone layers to unlock in Phase-2

CLASS_NAMES = ["Bacterial Leaf Disease", "Dried Leaf",
               "Fungal Brown Spot Disease", "Healthy Leaf"]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   Device        : {DEVICE}")

In [ ]:
# ============================================================
# CELL 4: Offline Augmentation (run ONCE — before training)
# Creates a balanced dataset at AUGMENTED_PATH
# ============================================================

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

def augment_class_folder(src_folder: Path, dst_folder: Path, target: int):
    """
    Copy originals + generate synthetic images until we reach `target`.
    dst_folder is created fresh each run (idempotent).
    """
    dst_folder.mkdir(parents=True, exist_ok=True)

    # Collect original images
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    originals = [p for p in src_folder.iterdir() if p.suffix.lower() in exts]

    if not originals:
        print(f"  ⚠️  No images found in {src_folder} — skipping.")
        return 0

    count = 0

    # 1. Copy originals first
    for p in originals:
        shutil.copy(p, dst_folder / p.name)
        count += 1

    # 2. Augment until target is reached
    idx = 0
    while count < target:
        src_img_path = originals[idx % len(originals)]
        img = Image.open(src_img_path).convert("RGB")
        aug_img = aug_transform(img)
        save_name = f"aug_{count:05d}.jpg"
        aug_img.save(dst_folder / save_name)
        count += 1
        idx += 1

    return count


print("=" * 55)
print("OFFLINE AUGMENTATION  (target = {} images/class)".format(TARGET_PER_CLASS))
print("=" * 55)

total_generated = 0
for cls in CLASS_NAMES:
    src = DATASET_PATH / cls
    dst = AUGMENTED_PATH / cls

    if not src.exists():
        print(f"  ❌ Folder not found: {src}")
        continue

    n_orig = len([f for f in src.iterdir()
                  if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tiff",".webp"}])
    print(f"\n  [{cls}]  originals: {n_orig}", end=" → ")

    n_final = augment_class_folder(src, dst, TARGET_PER_CLASS)
    print(f"total after augmentation: {n_final}")
    total_generated += n_final

print(f"\n✅ Augmentation complete.  Grand total images: {total_generated}")

In [ ]:
# ============================================================
# CELL 5: Train / Validation Split  (stratified, no augmentation)
# ============================================================

def stratified_split(src_root: Path, dst_root: Path, val_frac: float, seed: int = SEED):
    """
    Copies files from src_root/<class>/ into
    dst_root/train/<class>/ and dst_root/val/<class>/
    """
    random.seed(seed)
    summary = {}

    for cls_dir in sorted(src_root.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name
        all_imgs = sorted(cls_dir.glob("*.*"))
        random.shuffle(all_imgs)

        n_val   = int(len(all_imgs) * val_frac)
        val_imgs = all_imgs[:n_val]
        trn_imgs = all_imgs[n_val:]

        for split, imgs in [("train", trn_imgs), ("val", val_imgs)]:
            out_dir = dst_root / split / cls_name
            out_dir.mkdir(parents=True, exist_ok=True)
            for p in imgs:
                shutil.copy(p, out_dir / p.name)

        summary[cls_name] = {"train": len(trn_imgs), "val": len(val_imgs)}

    return summary


print("Splitting into train / val …")
split_info = stratified_split(AUGMENTED_PATH, FINAL_PATH, VAL_SPLIT)

print("\n{:<30} {:>8} {:>8}".format("Class", "Train", "Val"))
print("-" * 50)
for cls, counts in split_info.items():
    print("{:<30} {:>8} {:>8}".format(cls, counts["train"], counts["val"]))

total_train = sum(v["train"] for v in split_info.values())
total_val   = sum(v["val"]   for v in split_info.values())
print("-" * 50)
print("{:<30} {:>8} {:>8}".format("TOTAL", total_train, total_val))
print("\n✅ Split complete.")

In [ ]:
# ============================================================
# CELL 6: DataLoaders  (NO augmentation transforms — already done)
# ============================================================

# EfficientNet ImageNet normalisation
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder(FINAL_PATH / "train", transform=train_transform)
val_dataset   = datasets.ImageFolder(FINAL_PATH / "val",   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Class mapping      : {train_dataset.class_to_idx}")

In [ ]:
# ============================================================
# CELL 7: EfficientNetB0 Model with Custom Head
# ============================================================

def build_model(num_classes: int, device: torch.device) -> nn.Module:
    """
    EfficientNetB0 backbone + custom classification head.
    All backbone weights frozen initially (Phase-1 head training).
    """
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = efficientnet_b0(weights=weights)

    # ── Freeze entire backbone ──────────────────────────────
    for param in model.parameters():
        param.requires_grad = False

    # ── Replace classifier head ─────────────────────────────
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )

    return model.to(device)


model = build_model(NUM_CLASSES, DEVICE)

# Print parameter counts
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable (head)    : {trainable_params:,}")

In [ ]:
# ============================================================
# CELL 8: Training Utilities
# ============================================================

class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4, restore_best=True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.restore_best = restore_best
        self.best_loss    = float("inf")
        self.counter      = 0
        self.best_weights = None
        self.stop         = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            if self.restore_best:
                self.best_weights = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                if self.restore_best and self.best_weights:
                    model.load_state_dict(
                        {k: v.to(DEVICE) for k, v in self.best_weights.items()}
                    )


def run_epoch(model, loader, criterion, optimizer, phase="train"):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            running_loss    += loss.item() * imgs.size(0)
            running_correct += (preds == labels).sum().item()
            total           += imgs.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = running_correct / total
    return epoch_loss, epoch_acc


def unfreeze_top_layers(model, unfreeze_pct=0.30):
    """Unfreeze the last `unfreeze_pct` fraction of backbone feature layers."""
    feature_layers = list(model.features.children())
    n_unlock       = max(1, int(len(feature_layers) * unfreeze_pct))
    layers_to_unlock = feature_layers[-n_unlock:]

    for layer in layers_to_unlock:
        for param in layer.parameters():
            param.requires_grad = True

    unlocked = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total    = sum(p.numel() for p in model.parameters())
    print(f"  Unlocked last {n_unlock}/{len(feature_layers)} feature blocks")
    print(f"  Trainable params: {unlocked:,} / {total:,} "
          f"({100*unlocked/total:.1f}%)")


print("✅ Training utilities ready.")

In [ ]:
# ============================================================
# CELL 9: Phase 1 — Train classifier head only (backbone frozen)
# ============================================================

PHASE1_EPOCHS = NUM_EPOCHS  # up to 30; early stopping will kick in
criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_p1  = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler_p1  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=PHASE1_EPOCHS, eta_min=1e-6
)
early_stop_p1 = EarlyStopping(patience=PATIENCE, restore_best=True)

history_p1 = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

print("=" * 60)
print("PHASE 1: HEAD TRAINING  (backbone frozen)")
print("=" * 60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer_p1, "train")
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None,          "val")
    scheduler_p1.step()

    history_p1["train_loss"].append(tr_loss)
    history_p1["train_acc"].append(tr_acc)
    history_p1["val_loss"].append(vl_loss)
    history_p1["val_acc"].append(vl_acc)

    elapsed = time.time() - t0
    print(f"Ep {epoch:02d}/{PHASE1_EPOCHS}  "
          f"T-loss:{tr_loss:.4f}  T-acc:{tr_acc:.4f}  |  "
          f"V-loss:{vl_loss:.4f}  V-acc:{vl_acc:.4f}  "
          f"[{elapsed:.1f}s]")

    early_stop_p1(vl_loss, model)
    if early_stop_p1.stop:
        print(f"\n⏹  Early stopping at epoch {epoch} "
              f"(best val-loss: {early_stop_p1.best_loss:.4f})")
        break

print("\n✅ Phase 1 complete.")
torch.save(model.state_dict(), "/kaggle/working/efficientnet_b0_phase1.pth")
print("   Checkpoint saved → efficientnet_b0_phase1.pth")

In [ ]:
# ============================================================
# CELL 10: Phase 2 — Fine-tune (partial backbone unlock)
# ============================================================

print("=" * 60)
print(f"PHASE 2: FINE-TUNING  (unlock top {UNFREEZE_PCT*100:.0f}% of backbone)")
print("=" * 60)

unfreeze_top_layers(model, unfreeze_pct=UNFREEZE_PCT)

# Differential LRs: lower for backbone, higher for head
param_groups = [
    {"params": [p for n, p in model.named_parameters()
                if "classifier" not in n and p.requires_grad],
     "lr": LR_FINE},
    {"params": model.classifier.parameters(),
     "lr": LR_FINE * 5},
]

PHASE2_EPOCHS = 40
optimizer_p2  = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler_p2  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p2, T_0=10, T_mult=1, eta_min=1e-7
)
early_stop_p2 = EarlyStopping(patience=PATIENCE, restore_best=True)

history_p2 = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer_p2, "train")
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None,          "val")
    scheduler_p2.step()

    history_p2["train_loss"].append(tr_loss)
    history_p2["train_acc"].append(tr_acc)
    history_p2["val_loss"].append(vl_loss)
    history_p2["val_acc"].append(vl_acc)

    elapsed = time.time() - t0
    print(f"Ep {epoch:02d}/{PHASE2_EPOCHS}  "
          f"T-loss:{tr_loss:.4f}  T-acc:{tr_acc:.4f}  |  "
          f"V-loss:{vl_loss:.4f}  V-acc:{vl_acc:.4f}  "
          f"[{elapsed:.1f}s]")

    early_stop_p2(vl_loss, model)
    if early_stop_p2.stop:
        print(f"\n⏹  Early stopping at epoch {epoch} "
              f"(best val-loss: {early_stop_p2.best_loss:.4f})")
        break

print("\n✅ Phase 2 complete.")
torch.save(model.state_dict(), "/kaggle/working/efficientnet_b0_final.pth")
print("   Final checkpoint saved → efficientnet_b0_final.pth")

In [ ]:
# ============================================================
# CELL 11: Accuracy & Loss Curves
# ============================================================

def plot_history(hist, title_prefix=""):
    epochs_p1 = len(hist["p1"]["train_loss"])
    epochs_p2 = len(hist["p2"]["train_loss"])

    train_loss = hist["p1"]["train_loss"] + hist["p2"]["train_loss"]
    val_loss   = hist["p1"]["val_loss"]   + hist["p2"]["val_loss"]
    train_acc  = hist["p1"]["train_acc"]  + hist["p2"]["train_acc"]
    val_acc    = hist["p1"]["val_acc"]    + hist["p2"]["val_acc"]
    ep_x = range(1, len(train_loss) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    phase2_start = epochs_p1 + 0.5

    for ax, metric, ylabel in zip(
        axes,
        [(train_loss, val_loss), (train_acc, val_acc)],
        ["Loss", "Accuracy"]
    ):
        tr_vals, vl_vals = metric
        ax.plot(ep_x, tr_vals, label="Train",      color="#2196F3", lw=2)
        ax.plot(ep_x, vl_vals, label="Validation", color="#FF5722", lw=2)
        ax.axvline(x=phase2_start, color="green", linestyle="--",
                   alpha=0.7, label="Phase 2 start")
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"{title_prefix}{ylabel} Curve", fontsize=14, fontweight="bold")
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved → training_curves.png")


combined_history = {"p1": history_p1, "p2": history_p2}
plot_history(combined_history, title_prefix="Betel Leaf CNN — ")

In [ ]:
# ============================================================
# CELL 12: Classification Report
# ============================================================

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Map indices back to class names in correct order
idx_to_cls = {v: k for k, v in val_dataset.class_to_idx.items()}
target_names = [idx_to_cls[i] for i in range(NUM_CLASSES)]

print("=" * 65)
print("CLASSIFICATION REPORT")
print("=" * 65)
print(classification_report(all_labels, all_preds,
                             target_names=target_names, digits=4))

In [ ]:
# ============================================================
# CELL 13: Confusion Matrix
# ============================================================

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Raw counts ─────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Predicted Label", fontsize=11)
axes[0].set_ylabel("True Label", fontsize=11)
axes[0].tick_params(axis="x", rotation=30)
axes[0].tick_params(axis="y", rotation=0)

# ── Normalised (row %) ──────────────────────────────────────
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Confusion Matrix (row-normalised)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Predicted Label", fontsize=11)
axes[1].set_ylabel("True Label", fontsize=11)
axes[1].tick_params(axis="x", rotation=30)
axes[1].tick_params(axis="y", rotation=0)

plt.suptitle("Betel Leaf Disease — EfficientNetB0", fontsize=16,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → confusion_matrix.png")

In [ ]:
# ============================================================
# CELL 14: Feature Extractor — hook penultimate layer
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from pathlib import Path

# Load the trained model
model.eval()
model.to(DEVICE)

# ── Hook to extract features before the classifier head ────
# EfficientNetB0: model.avgpool outputs (B, 1280, 1, 1)
# We flatten that to (B, 1280) as the feature vector

extracted_features = {}

def hook_fn(module, input, output):
    extracted_features["feat"] = output.squeeze(-1).squeeze(-1)  # (B, 1280)

# Register hook on adaptive average pool
hook_handle = model.avgpool.register_forward_hook(hook_fn)

print("✅ Feature extractor hook registered on model.avgpool")
print(f"   Feature dimensionality: 1280")

In [ ]:
# ============================================================
# CELL 15: Extract features from TRAINING set (per class)
# These are used to compute class-conditional Gaussians
# ============================================================

# Use same val_transform (no augmentation) on training set
train_feat_dataset = datasets.ImageFolder(
    FINAL_PATH / "train", transform=val_transform
)
train_feat_loader = DataLoader(
    train_feat_dataset, batch_size=64,
    shuffle=False, num_workers=2, pin_memory=True
)

idx_to_cls = {v: k for k, v in train_feat_dataset.class_to_idx.items()}
print(f"Class mapping: {train_feat_dataset.class_to_idx}\n")

# Collect features and labels
all_feats  = []
all_labels = []

model.eval()
with torch.no_grad():
    for imgs, labels in train_feat_loader:
        imgs = imgs.to(DEVICE)
        _    = model(imgs)                         # triggers hook
        feats = extracted_features["feat"].cpu()   # (B, 1280)
        all_feats.append(feats)
        all_labels.append(labels)

all_feats  = torch.cat(all_feats,  dim=0).numpy()  # (N, 1280)
all_labels = torch.cat(all_labels, dim=0).numpy()  # (N,)

print(f"Total training features: {all_feats.shape}")

# ── Compute per-class mean vectors ─────────────────────────
class_means = {}
class_feats  = {}

for cls_idx in range(NUM_CLASSES):
    mask = (all_labels == cls_idx)
    feats_c = all_feats[mask]                # (Nc, 1280)
    class_feats[cls_idx]  = feats_c
    class_means[cls_idx]  = feats_c.mean(axis=0)
    print(f"  Class {cls_idx} [{idx_to_cls[cls_idx]:<30}]: {feats_c.shape[0]} samples")

print("\n✅ Per-class means computed.")

In [ ]:
# ============================================================
# CELL 16: Shared Tied Covariance + Precision Matrix
# Using tied (shared) covariance is standard for Mahalanobis
# OOD detection and avoids per-class singularity issues.
# ============================================================
from sklearn.covariance import EmpiricalCovariance, LedoitWolf

# Center features per class before pooling
centered_feats = []
for cls_idx in range(NUM_CLASSES):
    diff = class_feats[cls_idx] - class_means[cls_idx]
    centered_feats.append(diff)

centered_all = np.vstack(centered_feats)   # (N, 1280)
print(f"Centered features shape: {centered_all.shape}")

# ── Ledoit-Wolf shrinkage estimator ────────────────────────
# LedoitWolf is numerically stable for high-dim, small-N cases
# (1280-dim features with only ~1600 train samples per class)
print("\nFitting Ledoit-Wolf covariance estimator …")
lw = LedoitWolf(assume_centered=True)
lw.fit(centered_all)

# Precision matrix = inverse of covariance
cov_matrix  = lw.covariance_   # (1280, 1280)
prec_matrix = lw.precision_    # (1280, 1280)  — pre-inverted

print(f"Covariance matrix shape : {cov_matrix.shape}")
print(f"Precision matrix shape  : {prec_matrix.shape}")
print(f"LW shrinkage coefficient: {lw.shrinkage_:.4f}")
print("\n✅ Covariance & precision matrices ready.")

# Convert to torch for fast GPU batched computation
class_means_t = {k: torch.tensor(v, dtype=torch.float32).to(DEVICE)
                 for k, v in class_means.items()}
prec_t = torch.tensor(prec_matrix, dtype=torch.float32).to(DEVICE)

In [ ]:
# ============================================================
# CELL 17: Mahalanobis Distance Scorer
# Score = min over all classes of M-distance²
# Lower  → more IN-distribution
# Higher → more OUT-of-distribution
# ============================================================

def mahalanobis_distance_batch(features: torch.Tensor,
                                mean: torch.Tensor,
                                precision: torch.Tensor) -> torch.Tensor:
    """
    Computes squared Mahalanobis distance for a batch.
    
    D²(x, μ) = (x - μ)ᵀ Σ⁻¹ (x - μ)
    
    Args:
        features  : (B, D) feature vectors
        mean      : (D,)   class mean
        precision : (D, D) precision matrix (Σ⁻¹)
    Returns:
        distances : (B,)   squared M-distances
    """
    diff = features - mean.unsqueeze(0)             # (B, D)
    # Efficient computation: diag( diff @ Σ⁻¹ @ diffᵀ )
    left = diff @ precision                          # (B, D)
    dist = (left * diff).sum(dim=1)                  # (B,)
    return dist


def get_mahalanobis_scores(loader: DataLoader,
                            class_means_t: dict,
                            prec_t: torch.Tensor,
                            n_classes: int) -> tuple:
    """
    Returns:
        scores : (N,) minimum Mahalanobis distance across classes  
                 (OOD score — higher = more OOD)
        labels : (N,) true class labels
        preds  : (N,) model predicted class
    """
    all_scores, all_labels, all_preds = [], [], []

    model.eval()
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            feats  = extracted_features["feat"]     # (B, 1280)

            # Compute M-distance to every class mean
            dists = torch.stack([
                mahalanobis_distance_batch(feats, class_means_t[c], prec_t)
                for c in range(n_classes)
            ], dim=1)                                # (B, n_classes)

            min_dist, _ = dists.min(dim=1)           # (B,) — OOD score
            preds = logits.argmax(dim=1)

            all_scores.append(min_dist.cpu())
            all_labels.append(labels)
            all_preds.append(preds.cpu())

    scores = torch.cat(all_scores).numpy()
    labels = torch.cat(all_labels).numpy()
    preds  = torch.cat(all_preds).numpy()
    return scores, labels, preds


print("✅ Mahalanobis scorer defined.")

In [ ]:
# ============================================================
# CELL 18: Calibrate OOD Threshold on Validation Set
# We use the 95th percentile of IN-distribution scores
# as the decision boundary.
# ============================================================
from sklearn.preprocessing import StandardScaler

print("Scoring validation set (IN-distribution) …")
val_scores, val_labels, val_preds = get_mahalanobis_scores(
    val_loader, class_means_t, prec_t, NUM_CLASSES
)

print(f"\nIN-distribution score statistics:")
print(f"  Min   : {val_scores.min():.4f}")
print(f"  Max   : {val_scores.max():.4f}")
print(f"  Mean  : {val_scores.mean():.4f}")
print(f"  Std   : {val_scores.std():.4f}")
print(f"  p50   : {np.percentile(val_scores, 50):.4f}")
print(f"  p90   : {np.percentile(val_scores, 90):.4f}")
print(f"  p95   : {np.percentile(val_scores, 95):.4f}")
print(f"  p99   : {np.percentile(val_scores, 99):.4f}")

# ── Set threshold at 95th percentile ───────────────────────
OOD_THRESHOLD = np.percentile(val_scores, 95)
print(f"\n📌 OOD Threshold (95th pct): {OOD_THRESHOLD:.4f}")
print("   Samples with score > threshold → flagged as OOD")

# Save threshold for inference
np.save("/kaggle/working/ood_threshold.npy", OOD_THRESHOLD)
np.save("/kaggle/working/class_means.npy",
        np.stack([class_means[i] for i in range(NUM_CLASSES)]))
np.save("/kaggle/working/precision_matrix.npy", prec_matrix)
print("\n✅ Threshold & statistics saved.")

In [ ]:
# ============================================================
# CELL 19: Simulate OOD Data & Evaluate OOD Detector
# We simulate OOD by applying heavy corruptions to val images.
# In production, replace with a genuinely OOD dataset.
# ============================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class OODSimulatedDataset(Dataset):
    """
    Wraps a real dataset and applies heavy OOD corruptions:
    Gaussian noise + extreme color jitter + random erasing.
    These push samples far from the learned class manifolds.
    """
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.ood_transform = transforms.Compose([
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=15, sigma=(5.0, 10.0))
            ], p=0.8),
            transforms.ColorJitter(brightness=2.0, contrast=2.0,
                                   saturation=3.0, hue=0.5),
            transforms.RandomGrayscale(p=0.5),
            transforms.RandomErasing(p=1.0, scale=(0.3, 0.6),
                                     ratio=(0.3, 3.3), value="random"),
        ])

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img_tensor, label = self.base[idx]
        # Add Gaussian noise on top of corruptions
        noise = torch.randn_like(img_tensor) * 0.5
        img_tensor = torch.clamp(img_tensor + noise, -3, 3)
        img_tensor = self.ood_transform(img_tensor)
        return img_tensor, label


# Build OOD dataset (use val set as base)
ood_base    = datasets.ImageFolder(FINAL_PATH / "val", transform=val_transform)
ood_dataset = OODSimulatedDataset(ood_base)
ood_loader  = DataLoader(ood_dataset, batch_size=64,
                         shuffle=False, num_workers=2, pin_memory=True)

print("Scoring OOD (simulated corrupted) samples …")
ood_scores, _, _ = get_mahalanobis_scores(
    ood_loader, class_means_t, prec_t, NUM_CLASSES
)

print(f"\nOOD score statistics:")
print(f"  Min   : {ood_scores.min():.4f}")
print(f"  Max   : {ood_scores.max():.4f}")
print(f"  Mean  : {ood_scores.mean():.4f}")
print(f"  Std   : {ood_scores.std():.4f}")

# ── Binary OOD labels (0=IN, 1=OOD) ───────────────────────
y_true = np.concatenate([
    np.zeros(len(val_scores)),   # IN = 0
    np.ones(len(ood_scores))     # OOD = 1
])
y_score = np.concatenate([val_scores, ood_scores])
y_pred  = (y_score > OOD_THRESHOLD).astype(int)

from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)

auroc = roc_auc_score(y_true, y_score)
aupr  = average_precision_score(y_true, y_score)

print(f"\n{'='*45}")
print("OOD DETECTION PERFORMANCE")
print(f"{'='*45}")
print(f"  AUROC   : {auroc:.4f}")
print(f"  AUPR    : {aupr:.4f}")
print(f"  Threshold: {OOD_THRESHOLD:.4f}")

tp = ((y_pred == 1) & (y_true == 1)).sum()
tn = ((y_pred == 0) & (y_true == 0)).sum()
fp = ((y_pred == 1) & (y_true == 0)).sum()
fn = ((y_pred == 0) & (y_true == 1)).sum()

print(f"\n  True Positives  (OOD correctly flagged) : {tp}")
print(f"  True Negatives  (IN correctly passed)   : {tn}")
print(f"  False Positives (IN wrongly flagged)    : {fp}")
print(f"  False Negatives (OOD missed)            : {fn}")
print(f"  TPR @ 95th pct threshold                : {tp/(tp+fn):.4f}")
print(f"  FPR @ 95th pct threshold                : {fp/(fp+tn):.4f}")

print("\n✅ OOD evaluation complete.")

In [ ]:
# ============================================================
# CELL 20: Plot OOD Score Distributions
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── 1. Score distributions (KDE) ───────────────────────────
ax = axes[0]
ax.hist(val_scores, bins=60, alpha=0.65, color="#2196F3",
        density=True, label="IN-distribution (val)")
ax.hist(ood_scores, bins=60, alpha=0.65, color="#FF5722",
        density=True, label="OOD (simulated)")
ax.axvline(OOD_THRESHOLD, color="green", lw=2.5, linestyle="--",
           label=f"Threshold = {OOD_THRESHOLD:.1f}")
ax.set_xlabel("Mahalanobis Distance Score", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("IN vs OOD Score Distributions", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── 2. ROC Curve ───────────────────────────────────────────
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_score)

ax = axes[1]
ax.plot(fpr, tpr, color="#9C27B0", lw=2.5, label=f"AUROC = {auroc:.4f}")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random")
ax.fill_between(fpr, tpr, alpha=0.08, color="#9C27B0")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve (OOD Detection)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# ── 3. Precision-Recall Curve ──────────────────────────────
from sklearn.metrics import precision_recall_curve
prec_c, rec_c, _ = precision_recall_curve(y_true, y_score)

ax = axes[2]
ax.plot(rec_c, prec_c, color="#FF9800", lw=2.5,
        label=f"AUPR = {aupr:.4f}")
ax.axhline(y_true.mean(), color="gray", lw=1, linestyle="--",
           label=f"Baseline = {y_true.mean():.3f}")
ax.fill_between(rec_c, prec_c, alpha=0.08, color="#FF9800")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle("Mahalanobis OOD Detection — Betel Leaf EfficientNetB0",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/ood_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → ood_distributions.png")

In [ ]:
# ============================================================
# CELL 21: Per-Class Distance Heatmap
# Shows how far each val sample is from every class center.
# ============================================================
model.eval()
all_class_dists = []
all_true_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        _ = model(imgs)
        feats = extracted_features["feat"]

        dists = torch.stack([
            mahalanobis_distance_batch(feats, class_means_t[c], prec_t)
            for c in range(NUM_CLASSES)
        ], dim=1).cpu().numpy()

        all_class_dists.append(dists)
        all_true_labels.append(labels.numpy())

all_class_dists = np.vstack(all_class_dists)   # (N, 4)
all_true_labels = np.concatenate(all_true_labels)

# Compute mean per-class distance matrix
mean_dist_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES))
for true_c in range(NUM_CLASSES):
    mask = (all_true_labels == true_c)
    mean_dist_matrix[true_c] = all_class_dists[mask].mean(axis=0)

short_names = ["Bacterial", "Dried", "Fungal", "Healthy"]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(mean_dist_matrix, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax, label="Mean Mahalanobis Distance²")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(short_names, rotation=30, ha="right")
ax.set_yticklabels(short_names)
ax.set_xlabel("Distance to Class Center", fontsize=12)
ax.set_ylabel("True Class", fontsize=12)
ax.set_title("Per-Class Mahalanobis Distance Matrix\n(Diagonal = IN-class distance; off-diagonal = cross-class)",
             fontsize=13, fontweight="bold")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{mean_dist_matrix[i, j]:.1f}",
                ha="center", va="center", fontsize=11,
                color="black" if mean_dist_matrix[i, j] < mean_dist_matrix.max() * 0.6 else "white")

plt.tight_layout()
plt.savefig("/kaggle/working/mahalanobis_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → mahalanobis_heatmap.png")

In [ ]:
# ============================================================
# CELL 22: Full Inference Wrapper with OOD Rejection
# Drop-in function for production / test-time use
# ============================================================

def predict_with_ood(image_tensor: torch.Tensor,
                     threshold: float = OOD_THRESHOLD) -> dict:
    """
    Run inference + OOD detection on a single image (or batch).

    Args:
        image_tensor : (C, H, W) or (B, C, H, W) preprocessed tensor
        threshold    : Mahalanobis score cutoff (default: calibrated 95th pct)

    Returns dict with:
        - predicted_class  : class name (or "OOD" if rejected)
        - confidence       : softmax probability of predicted class
        - ood_score        : Mahalanobis distance (lower = more in-distribution)
        - is_ood           : bool flag
        - class_scores     : dict of {class: confidence} for all classes
    """
    model.eval()
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)    # (1, C, H, W)
    image_tensor = image_tensor.to(DEVICE)

    with torch.no_grad():
        logits = model(image_tensor)                # also triggers hook
        feats  = extracted_features["feat"]         # (B, 1280)

        # Compute min Mahalanobis distance over all classes
        dists = torch.stack([
            mahalanobis_distance_batch(feats, class_means_t[c], prec_t)
            for c in range(NUM_CLASSES)
        ], dim=1)                                   # (B, n_classes)

        min_dist, _ = dists.min(dim=1)
        ood_score   = min_dist.cpu().item()

        probs       = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx    = int(probs.argmax())
        confidence  = float(probs[pred_idx])
        is_ood      = ood_score > threshold

    return {
        "predicted_class" : "OOD — Rejected" if is_ood else idx_to_cls[pred_idx],
        "confidence"      : round(confidence, 4) if not is_ood else None,
        "ood_score"       : round(ood_score, 4),
        "threshold"       : round(threshold, 4),
        "is_ood"          : is_ood,
        "class_scores"    : {idx_to_cls[i]: round(float(probs[i]), 4)
                             for i in range(NUM_CLASSES)},
    }


# ── Quick demo on a few validation samples ─────────────────
print("=" * 55)
print("INFERENCE DEMO WITH OOD REJECTION")
print("=" * 55)

demo_dataset = datasets.ImageFolder(FINAL_PATH / "val", transform=val_transform)
demo_loader  = DataLoader(demo_dataset, batch_size=1, shuffle=True)

for i, (img, label) in enumerate(demo_loader):
    if i >= 6:
        break
    result    = predict_with_ood(img.squeeze(0))
    true_name = idx_to_cls[label.item()]
    status    = "✅ IN" if not result["is_ood"] else "🚨 OOD"
    print(f"\n[Sample {i+1}]  True: {true_name:<30}")
    print(f"  Prediction : {result['predicted_class']}")
    print(f"  Confidence : {result['confidence']}")
    print(f"  OOD Score  : {result['ood_score']}  (threshold: {result['threshold']})")
    print(f"  Status     : {status}")

# ── Remove hook when done ───────────────────────────────────
hook_handle.remove()
print("\n✅ Hook removed. OOD system fully operational.")

In [ ]:
# ============================================================
# CELL: Export Flutter Integration Files (ONNX-based)
# ============================================================
import json
import zipfile
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn

EXPORT_DIR = Path("/kaggle/working/flutter_export")
EXPORT_DIR.mkdir(exist_ok=True)

# ── 1. Export Full Classification Model → ONNX ─────────────
print("Step 1: Exporting classification model to ONNX...")

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    str(EXPORT_DIR / "betel_model.onnx"),
    export_params=True,
    opset_version=12,          # opset 12 is most stable across converters
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    dynamo=False,              # force legacy exporter — avoids onnxscript bug
)
size = (EXPORT_DIR / "betel_model.onnx").stat().st_size / 1e6
print(f"   ✅ betel_model.onnx saved ({size:.2f} MB)")

# ── 2. Export Feature Extractor Model → ONNX ───────────────
# Wraps backbone + avgpool only (no classifier head)
# Output: (B, 1280) feature vector — used for Mahalanobis OOD
print("\nStep 2: Exporting feature extractor to ONNX...")

class FeatureExtractor(nn.Module):
    """EfficientNetB0 up to avgpool — outputs 1280-dim feature vector."""
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features
        self.avgpool  = backbone.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)       # (B, 1280)
        return x

feat_extractor = FeatureExtractor(model).to(DEVICE)
feat_extractor.eval()

torch.onnx.export(
    feat_extractor,
    dummy_input,
    str(EXPORT_DIR / "feature_extractor.onnx"),
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["features"],
    dynamic_axes={"input": {0: "batch"}, "features": {0: "batch"}},
    dynamo=False,
)
size = (EXPORT_DIR / "feature_extractor.onnx").stat().st_size / 1e6
print(f"   ✅ feature_extractor.onnx saved ({size:.2f} MB)")

# ── 3. Export OOD Config JSON ──────────────────────────────
print("\nStep 3: Saving OOD config...")

class_means_array = np.stack(
    [class_means[i] for i in range(NUM_CLASSES)]
).astype(np.float32).tolist()   # (4, 1280) → nested list for JSON

ood_config = {
    "ood_threshold"      : float(OOD_THRESHOLD),
    "threshold_percentile": 95,
    "feature_dim"        : 1280,
    "num_classes"        : NUM_CLASSES,
    "img_size"           : IMG_SIZE,
    "normalization_mean" : [0.485, 0.456, 0.406],
    "normalization_std"  : [0.229, 0.224, 0.225],
    "class_means"        : class_means_array,        # (4, 1280) embedded
    "precision_matrix"   : prec_matrix.astype(       # (1280, 1280) embedded
                               np.float32).tolist(),
}

with open(EXPORT_DIR / "ood_config.json", "w") as f:
    json.dump(ood_config, f)

size = (EXPORT_DIR / "ood_config.json").stat().st_size / 1e6
print(f"   ✅ ood_config.json saved ({size:.2f} MB)")

# ── 4. Export Labels JSON ──────────────────────────────────
print("\nStep 4: Saving labels...")

labels_data = {
    "labels"         : CLASS_NAMES,
    "index_to_class" : {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "num_classes"    : NUM_CLASSES,
}

with open(EXPORT_DIR / "labels.json", "w") as f:
    json.dump(labels_data, f, indent=2)

print(f"   ✅ labels.json saved")

# ── 5. Bundle into ZIP ─────────────────────────────────────
print("\nStep 5: Creating ZIP bundle...")

zip_path = Path("/kaggle/working/flutter_onnx_export.zip")
files_to_zip = [
    "betel_model.onnx",
    "feature_extractor.onnx",
    "ood_config.json",
    "labels.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        fpath = EXPORT_DIR / fname
        zf.write(fpath, fname)
        size = fpath.stat().st_size / 1e6
        print(f"   + {fname:<35} {size:>6.2f} MB")

print(f"\n{'='*50}")
print(f"✅ ZIP created → {zip_path}")
print(f"   Total ZIP size: {zip_path.stat().st_size / 1e6:.2f} MB")
print(f"{'='*50}")

In [ ]:
!pip install -q onnxscript
!pip install -q onnxruntime

In [ ]:
# ============================================================
# CELL: Test Inference Using Exported ONNX Files + OOD
# ============================================================

import onnxruntime as ort
import numpy as np
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Install onnxruntime if not available ───────────────────
try:
    import onnxruntime as ort
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "onnxruntime", "--quiet"])
    import onnxruntime as ort

# ── Paths ──────────────────────────────────────────────────
EXPORT_DIR       = Path("/kaggle/working/flutter_export")
MODEL_ONNX       = EXPORT_DIR / "betel_model.onnx"
FEAT_ONNX        = EXPORT_DIR / "feature_extractor.onnx"
OOD_CONFIG_PATH  = EXPORT_DIR / "ood_config.json"
LABELS_PATH      = EXPORT_DIR / "labels.json"

# ── Change this to any image path you want to test ─────────
TEST_IMAGE_PATH  = "/kaggle/input/datasets/bapdesilva/testing-001/lions gate 8.jpg"

# ── Load configs ───────────────────────────────────────────
with open(LABELS_PATH)     as f: labels_cfg = json.load(f)
with open(OOD_CONFIG_PATH) as f: ood_cfg    = json.load(f)

CLASS_NAMES_   = labels_cfg["labels"]
OOD_THRESH     = ood_cfg["ood_threshold"]
MEAN           = ood_cfg["normalization_mean"]
STD            = ood_cfg["normalization_std"]
IMG_SIZE_      = ood_cfg["img_size"]
class_means_np = np.array(ood_cfg["class_means"],      dtype=np.float32)  # (4, 1280)
prec_np        = np.array(ood_cfg["precision_matrix"], dtype=np.float32)  # (1280, 1280)

print("✅ Configs loaded")
print(f"   Classes       : {CLASS_NAMES_}")
print(f"   OOD Threshold : {OOD_THRESH:.4f}")

# ── Load ONNX sessions ─────────────────────────────────────
providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
             if ort.get_device() == "GPU" else ["CPUExecutionProvider"])

sess_model = ort.InferenceSession(str(MODEL_ONNX), providers=providers)
sess_feat  = ort.InferenceSession(str(FEAT_ONNX),  providers=providers)

print(f"\n✅ ONNX sessions loaded")
print(f"   Execution provider: {sess_model.get_providers()[0]}")

# ── Preprocessing (mirrors training val_transform) ─────────
def preprocess(image_path: str) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    img = img.resize((IMG_SIZE_, IMG_SIZE_), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0           # (H, W, 3)
    arr = (arr - np.array(MEAN)) / np.array(STD)            # normalize
    arr = arr.transpose(2, 0, 1)                             # (3, H, W)
    arr = arr[np.newaxis, ...]                               # (1, 3, H, W)
    return arr.astype(np.float32)

# ── Mahalanobis distance (pure numpy) ──────────────────────
def mahalanobis_score(features: np.ndarray,
                      class_means: np.ndarray,
                      precision: np.ndarray) -> float:
    """
    Computes min squared Mahalanobis distance over all class means.
    features     : (1, 1280)
    class_means  : (4, 1280)
    precision    : (1280, 1280)
    Returns      : scalar OOD score (lower = more in-distribution)
    """
    dists = []
    for mu in class_means:
        diff = features[0] - mu                  # (1280,)
        left = diff @ precision                  # (1280,)
        d2   = float(left @ diff)                # scalar
        dists.append(d2)
    return float(min(dists))

# ── Run full inference pipeline ────────────────────────────
print(f"\n{'='*55}")
print(f"TEST IMAGE: {TEST_IMAGE_PATH}")
print(f"{'='*55}")

input_tensor = preprocess(TEST_IMAGE_PATH)
print(f"\nInput tensor shape: {input_tensor.shape}")

# Step A: Classification
logits   = sess_model.run(None, {"input": input_tensor})[0]  # (1, 4)
probs    = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
pred_idx = int(probs.argmax())
confidence = float(probs[0, pred_idx])

# Step B: Feature extraction
features = sess_feat.run(None, {"input": input_tensor})[0]   # (1, 1280)

# Step C: OOD scoring
ood_score = mahalanobis_score(features, class_means_np, prec_np)
is_ood    = ood_score > OOD_THRESH

# ── Print results ──────────────────────────────────────────
print(f"\n{'─'*45}")
print(f"  Predicted class : {CLASS_NAMES_[pred_idx]}")
print(f"  Confidence      : {confidence*100:.2f}%")
print(f"  OOD Score       : {ood_score:.4f}")
print(f"  OOD Threshold   : {OOD_THRESH:.4f}")
print(f"  Status          : {'🚨 OOD — REJECTED' if is_ood else '✅ IN-DISTRIBUTION'}")
print(f"{'─'*45}")
print(f"\n  Per-class probabilities:")
for i, (cls, prob) in enumerate(zip(CLASS_NAMES_, probs[0])):
    bar = "█" * int(prob * 30)
    marker = " ◄" if i == pred_idx else ""
    print(f"    {cls:<30} {prob*100:5.2f}%  {bar}{marker}")

# ── Visualize ──────────────────────────────────────────────
img_display = Image.open(TEST_IMAGE_PATH).convert("RGB")
fig, axes   = plt.subplots(1, 2, figsize=(14, 5))

# Left: image + result overlay
axes[0].imshow(img_display)
axes[0].axis("off")
status_color = "#FF5722" if is_ood else "#4CAF50"
status_text  = "🚨 OOD — REJECTED" if is_ood else "✅ IN-DISTRIBUTION"
axes[0].set_title(
    f"{status_text}\n"
    f"Pred: {CLASS_NAMES_[pred_idx]}  ({confidence*100:.1f}%)\n"
    f"OOD Score: {ood_score:.2f}  |  Threshold: {OOD_THRESH:.2f}",
    fontsize=12, fontweight="bold", color=status_color, pad=10
)

# Right: confidence bar chart
colors = ["#2196F3" if i == pred_idx else "#B0BEC5"
          for i in range(len(CLASS_NAMES_))]
short_names = [n.replace(" Disease", "").replace(" Leaf", " Leaf")
               for n in CLASS_NAMES_]
bars = axes[1].barh(short_names, probs[0] * 100, color=colors,
                    edgecolor="white", height=0.55)
axes[1].set_xlabel("Confidence (%)", fontsize=11)
axes[1].set_title("Class Probabilities", fontsize=13, fontweight="bold")
axes[1].set_xlim(0, 110)
axes[1].axvline(x=50, color="gray", linestyle="--", alpha=0.4)
axes[1].grid(axis="x", alpha=0.3)

for bar, prob in zip(bars, probs[0]):
    axes[1].text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
                 f"{prob*100:.1f}%", va="center", fontsize=11)

# OOD threshold annotation
ood_patch = mpatches.Patch(
    color=status_color,
    label=f"OOD Score: {ood_score:.2f}  (threshold: {OOD_THRESH:.2f})"
)
axes[1].legend(handles=[ood_patch], loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig("/kaggle/working/onnx_test_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Result saved → onnx_test_result.png")